In [29]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd

In [30]:
PROJECT_ROOT = Path.cwd().parent.parent.parent

EXTRACTED_DIR = (
    PROJECT_ROOT
    / "data"
    / "extracted"
    / "merged_cells"
)

MANIFEST_PATH = EXTRACTED_DIR / "manifest.csv"

print("Project root:")
print(PROJECT_ROOT)

print("\nExtracted-data directory:")
print(EXTRACTED_DIR)


Project root:
D:\Projects\Kaggle\cell-tracking

Extracted-data directory:
D:\Projects\Kaggle\cell-tracking\data\extracted\merged_cells


In [31]:
ARTIFACT_NAMES = (
    "raw",
    "preprocessed",
    "binary_mask",
    "instance_labels",
)


def discover_extracted_merged_cells(
    extracted_dir: Path,
) -> pd.DataFrame:
    """Discover only the merged-cell volume from every extracted case.

    Expected layout:

        merged_cells/<case_name>/merged_cell/metadata.json
        merged_cells/<case_name>/merged_cell/raw.npy
        merged_cells/<case_name>/merged_cell/preprocessed.npy
        merged_cells/<case_name>/merged_cell/binary_mask.npy
        merged_cells/<case_name>/merged_cell/instance_labels.npy
    """

    records = []

    metadata_files = sorted(
        extracted_dir.glob("*/merged_cell/metadata.json")
    )

    if not metadata_files:
        raise FileNotFoundError(
            "No extracted merged-cell cases were found.\n"
            f"Expected files matching:\n"
            f"{extracted_dir / '*/merged_cell/metadata.json'}"
        )

    for metadata_path in metadata_files:
        merged_cell_dir = metadata_path.parent
        case_dir = merged_cell_dir.parent
        case_metadata_path = case_dir / "metadata.json"

        with metadata_path.open("r", encoding="utf-8") as file:
            metadata = json.load(file)

        case_metadata = {}
        if case_metadata_path.exists():
            with case_metadata_path.open("r", encoding="utf-8") as file:
                case_metadata = json.load(file)

        artifact_files = metadata.get("artifact_files", {})

        record = {
            "case_name": case_dir.name,
            "sample_id": metadata.get(
                "sample_id",
                case_metadata.get("sample_id"),
            ),
            "frame": int(metadata["frame"]),
            "cell_id": int(metadata["cell_id"]),
            "stem": case_dir.name,
            "case_dir": case_dir,
            "merged_cell_dir": merged_cell_dir,
            "metadata_path": metadata_path,
            "case_metadata_path": case_metadata_path,
            "box_size_zyx": metadata.get("box_size_zyx"),
            "centroid_zyx": metadata.get("centroid_zyx"),
        }

        for artifact_name in ARTIFACT_NAMES:
            filename = artifact_files.get(
                artifact_name,
                f"{artifact_name}.npy",
            )
            artifact_path = merged_cell_dir / filename

            record[f"{artifact_name}_path"] = artifact_path
            record[f"has_{artifact_name}"] = artifact_path.exists()

        records.append(record)

    cases = pd.DataFrame(records)

    return cases.sort_values(
        ["sample_id", "frame", "cell_id"]
    ).reset_index(drop=True)


cases = discover_extracted_merged_cells(EXTRACTED_DIR)

cases[
    [
        "case_name",
        "sample_id",
        "frame",
        "cell_id",
        "box_size_zyx",
        "has_raw",
        "has_preprocessed",
        "has_binary_mask",
        "has_instance_labels",
    ]
]


,case_name,sample_id,frame,cell_id,box_size_zyx,has_raw,has_preprocessed,has_binary_mask,has_instance_labels
0,44b6_0113de3b__cell1_t006_id00118__cell2_t006_...,44b6_0113de3b,8,130,"[12, 50, 50]",True,True,True,True
1,44b6_0113de3b__cell1_t009_id00063__cell2_t009_...,44b6_0113de3b,10,79,"[12, 50, 50]",True,True,True,True
2,44b6_0113de3b__cell1_t010_id00071__cell2_t010_...,44b6_0113de3b,11,71,"[12, 50, 50]",True,True,True,True
3,44b6_0113de3b__cell1_t009_id00063__cell2_t009_...,44b6_0113de3b,12,86,"[12, 50, 50]",True,True,True,True
4,44b6_0113de3b__cell1_t015_id00042__cell2_t015_...,44b6_0113de3b,17,42,"[17, 50, 50]",True,True,True,True


In [32]:
CASE_INDEX = 2

selected_case = cases.iloc[CASE_INDEX]

print(
    f"Selected case {CASE_INDEX}: "
    f"sample={selected_case['sample_id']}, "
    f"frame={selected_case['frame']}, "
    f"cell_id={selected_case['cell_id']}"
)

Selected case 2: sample=44b6_0113de3b, frame=11, cell_id=71


In [33]:
def load_array_if_available(
        path: Path,
) -> np.ndarray | None:
    if not path.exists():
        return None

    return np.load(
        path,
        allow_pickle=False,
    )


with selected_case["metadata_path"].open(
        "r",
        encoding="utf-8",
) as file:
    metadata = json.load(file)


raw_volume = load_array_if_available(
    selected_case["raw_path"]
)

preprocessed_volume = load_array_if_available(
    selected_case["preprocessed_path"]
)

binary_mask_volume = load_array_if_available(
    selected_case["binary_mask_path"]
)

instance_labels_volume = load_array_if_available(
    selected_case["instance_labels_path"]
)

In [34]:
def describe_array(
        name: str,
        array: np.ndarray | None,
) -> None:
    if array is None:
        print(f"{name:18s}: not available")
        return

    print(
        f"{name:18s}: "
        f"shape={array.shape}, "
        f"dtype={array.dtype}, "
        f"min={array.min()}, "
        f"max={array.max()}"
    )


describe_array("Raw", raw_volume)
describe_array("Preprocessed", preprocessed_volume)
describe_array("Binary mask", binary_mask_volume)
describe_array("Instance labels", instance_labels_volume)

Raw               : shape=(12, 50, 50), dtype=uint16, min=149, max=2476
Preprocessed      : shape=(12, 50, 50), dtype=float32, min=0.0, max=0.9055934548377991
Binary mask       : shape=(12, 50, 50), dtype=uint8, min=0, max=1
Instance labels   : shape=(12, 50, 50), dtype=int32, min=0, max=94


In [35]:
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets

from IPython.display import display


max_label = int(instance_labels_volume.max())


def show_diagnostic_slice(z: int) -> None:
    fig, axes = plt.subplots(
        1,
        3,
        figsize=(15, 5),
        constrained_layout=True,
    )

    axes[0].imshow(
        preprocessed_volume[z],
        cmap="gray",
        vmin=0,
        vmax=1,
    )
    axes[0].set_title(f"Preprocessed — Z={z}")

    axes[1].imshow(
        binary_mask_volume[z],
        cmap="gray",
        vmin=0,
        vmax=1,
    )
    axes[1].set_title(f"Binary mask — Z={z}")

    axes[2].imshow(
        instance_labels_volume[z],
        cmap="nipy_spectral",
        vmin=0,
        vmax=max_label,
        interpolation="nearest",
    )
    axes[2].set_title(f"Instance labels — Z={z}")

    for axis in axes:
        axis.axis("off")

    plt.show()


z_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=preprocessed_volume.shape[0] - 1,
    step=1,
    description="Z slice",
    continuous_update=False,
)

interactive_view = widgets.interactive_output(
    show_diagnostic_slice,
    {"z": z_slider},
)

display(z_slider, interactive_view)

IntSlider(value=0, continuous_update=False, description='Z slice', max=11)

Output()

Current state

In [36]:
%gui qt

In [37]:
import napari
import numpy as np

VOXEL_SIZE_ZYX = tuple(
    metadata.get(
        "voxel_size_zyx",
        [1.625, 0.40625, 0.40625],
    )
)

viewer = napari.Viewer(ndisplay=3)

# ------------------------------------------------------------
# Original intensity volume
# ------------------------------------------------------------

raw_layer = viewer.add_image(
    raw_volume,
    name="Original intensity",
    scale=VOXEL_SIZE_ZYX,
    colormap="gray",
    opacity=1.0,
    rendering="mip",
)

# ------------------------------------------------------------
# Preprocessed intensity volume
# Hidden initially, but available from the layer panel.
# ------------------------------------------------------------

preprocessed_layer = viewer.add_image(
    preprocessed_volume,
    name="Preprocessed intensity",
    scale=VOXEL_SIZE_ZYX,
    colormap="gray",
    opacity=1.0,
    rendering="mip",
    visible=False,
)

# ------------------------------------------------------------
# Binary mask
# Hidden initially because it has already been inspected.
# ------------------------------------------------------------

binary_layer = viewer.add_labels(
    binary_mask_volume.astype(np.uint8),
    name="Binary mask",
    scale=VOXEL_SIZE_ZYX,
    opacity=0.7,
    visible=False,
)

# ------------------------------------------------------------
# Current 3D watershed result
# ------------------------------------------------------------

watershed_layer = viewer.add_labels(
    instance_labels_volume.astype(np.int32),
    name="Current watershed labels",
    scale=VOXEL_SIZE_ZYX,
    opacity=0.65,
    visible=True,
)

# ------------------------------------------------------------
# Optional: show only the selected merged instance
# ------------------------------------------------------------

target_label = int(metadata["cell_id"])

selected_instance_mask = (
        instance_labels_volume == target_label
).astype(np.uint8)

selected_instance_layer = viewer.add_labels(
    selected_instance_mask,
    name=f"Selected instance - cell {target_label}",
    scale=VOXEL_SIZE_ZYX,
    opacity=0.8,
    visible=False,
)

# ------------------------------------------------------------
# Initial visibility
# ------------------------------------------------------------

raw_layer.visible = True
preprocessed_layer.visible = False
binary_layer.visible = False
watershed_layer.visible = True
selected_instance_layer.visible = False

viewer.camera.angles = (45, 30, 135)
viewer.reset_view()

In [38]:
from scipy import ndimage as ndi
import numpy as np


# ------------------------------------------------------------
# Identify the selected merged instance
# ------------------------------------------------------------

target_label = int(metadata["cell_id"])

available_labels = np.unique(instance_labels_volume)

if target_label not in available_labels:
    # Fallback: use the label at the center of the extracted box.
    center_zyx = tuple(
        np.asarray(instance_labels_volume.shape) // 2
    )

    target_label = int(
        instance_labels_volume[center_zyx]
    )

    if target_label == 0:
        raise ValueError(
            "Could not identify the selected instance. "
            "The crop center belongs to the background."
        )

target_mask = instance_labels_volume == target_label

print("Target label:", target_label)
print("Target voxels:", int(target_mask.sum()))


# ------------------------------------------------------------
# Smooth intensity using a physical smoothing width
# ------------------------------------------------------------

SMOOTHING_SIGMA_PHYSICAL = 0.6  # micrometres

sigma_zyx = tuple(
    SMOOTHING_SIGMA_PHYSICAL / spacing
    for spacing in VOXEL_SIZE_ZYX
)

print("Gaussian sigma in voxels:", sigma_zyx)

intensity_smoothed = ndi.gaussian_filter(
    preprocessed_volume.astype(np.float32),
    sigma=sigma_zyx,
)


# ------------------------------------------------------------
# Compute the physical-space 3D gradient
# ------------------------------------------------------------

gradient_z, gradient_y, gradient_x = np.gradient(
    intensity_smoothed,
    *VOXEL_SIZE_ZYX,
)

gradient_magnitude = np.sqrt(
    gradient_z**2
    + gradient_y**2
    + gradient_x**2
)


# ------------------------------------------------------------
# Keep only gradient values inside the selected instance
# ------------------------------------------------------------

gradient_inside_instance = np.where(
    target_mask,
    gradient_magnitude,
    0.0,
)


# ------------------------------------------------------------
# Robust normalization for visualization
# ------------------------------------------------------------

foreground_gradient = gradient_inside_instance[target_mask]

lower, upper = np.percentile(
    foreground_gradient,
    [1.0, 99.5],
)

if upper > lower:
    gradient_normalized = (
                                  gradient_inside_instance - lower
                          ) / (upper - lower)

    gradient_normalized = np.clip(
        gradient_normalized,
        0.0,
        1.0,
    )
else:
    gradient_normalized = np.zeros_like(
        gradient_inside_instance,
        dtype=np.float32,
    )


print(
    "Gradient range:",
    float(gradient_magnitude.min()),
    "to",
    float(gradient_magnitude.max()),
)

Target label: 71
Target voxels: 1964
Gaussian sigma in voxels: (0.3692307692307692, 1.4769230769230768, 1.4769230769230768)
Gradient range: 0.0 to 0.3730809986591339


In [39]:
# Remove an older copy when rerunning this cell.
gradient_layer_name = "3D intensity gradient"

if gradient_layer_name in viewer.layers:
    viewer.layers.remove(
        viewer.layers[gradient_layer_name]
    )


gradient_layer = viewer.add_image(
    gradient_normalized,
    name=gradient_layer_name,
    scale=VOXEL_SIZE_ZYX,
    colormap="magma",
    rendering="mip",
    opacity=0.9,
    visible=True,
)


# Keep useful comparison layers available but hidden.
viewer.layers["Original intensity"].visible = False
viewer.layers["Preprocessed intensity"].visible = False
viewer.layers["Binary mask"].visible = False
viewer.layers["Current watershed labels"].visible = False

gradient_layer.visible = True

viewer.reset_view()

In [40]:
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display


def show_gradient_slices(
        z: int,
        y: int,
        x: int,
) -> None:
    fig, axes = plt.subplots(
        1,
        3,
        figsize=(16, 5),
        constrained_layout=True,
    )

    axes[0].imshow(
        gradient_normalized[z, :, :],
        cmap="magma",
        vmin=0,
        vmax=1,
    )
    axes[0].set_title(f"XY gradient — Z={z}")

    axes[1].imshow(
        gradient_normalized[:, y, :],
        cmap="magma",
        vmin=0,
        vmax=1,
        aspect="auto",
    )
    axes[1].set_title(f"XZ gradient — Y={y}")

    axes[2].imshow(
        gradient_normalized[:, :, x],
        cmap="magma",
        vmin=0,
        vmax=1,
        aspect="auto",
    )
    axes[2].set_title(f"YZ gradient — X={x}")

    for axis in axes:
        axis.axis("off")

    plt.show()


z_slider = widgets.IntSlider(
    value=gradient_normalized.shape[0] // 2,
    min=0,
    max=gradient_normalized.shape[0] - 1,
    step=1,
    description="Z",
    continuous_update=False,
)

y_slider = widgets.IntSlider(
    value=gradient_normalized.shape[1] // 2,
    min=0,
    max=gradient_normalized.shape[1] - 1,
    step=1,
    description="Y",
    continuous_update=False,
)

x_slider = widgets.IntSlider(
    value=gradient_normalized.shape[2] // 2,
    min=0,
    max=gradient_normalized.shape[2] - 1,
    step=1,
    description="X",
    continuous_update=False,
)

gradient_slice_view = widgets.interactive_output(
    show_gradient_slices,
    {
        "z": z_slider,
        "y": y_slider,
        "x": x_slider,
    },
)

display(
    widgets.VBox(
        [
            z_slider,
            y_slider,
            x_slider,
        ]
    ),
    gradient_slice_view,
)

Output()

In [41]:
from scipy import ndimage as ndi
from skimage.morphology import h_maxima
from skimage.segmentation import watershed

import numpy as np


# ------------------------------------------------------------
# Parameters
# ------------------------------------------------------------

DISTANCE_SIGMA_PHYSICAL = 0.8  # micrometres

voxel_size = np.asarray(
    VOXEL_SIZE_ZYX,
    dtype=np.float64,
)

DISTANCE_SIGMA_ZYX = tuple(
    DISTANCE_SIGMA_PHYSICAL / voxel_size
)

# Because the distance transform uses physical sampling,
# h is also measured in the same physical distance units.
H_MAXIMA_HEIGHT = 0.5


print(
    "Distance smoothing sigma in voxels:",
    tuple(round(value, 3) for value in DISTANCE_SIGMA_ZYX),
)


# ------------------------------------------------------------
# Validate the selected component
# ------------------------------------------------------------

target_mask = np.asarray(
    target_mask,
    dtype=bool,
)

if target_mask.ndim != 3:
    raise ValueError(
        f"Expected a 3D target mask, found {target_mask.shape}."
    )

if not np.any(target_mask):
    raise ValueError("The selected target mask is empty.")


# ------------------------------------------------------------
# 1. Physical 3D distance transform
# ------------------------------------------------------------

distance_3d = ndi.distance_transform_edt(
    target_mask,
    sampling=VOXEL_SIZE_ZYX,
)


# ------------------------------------------------------------
# 2. Physically isotropic Gaussian smoothing
# ------------------------------------------------------------

distance_smooth = ndi.gaussian_filter(
    distance_3d,
    sigma=DISTANCE_SIGMA_ZYX,
)

distance_smooth[~target_mask] = 0.0


# ------------------------------------------------------------
# 3. Automatic H-maxima markers
# ------------------------------------------------------------

distance_hmax = h_maxima(
    distance_smooth,
    h=H_MAXIMA_HEIGHT,
)

distance_hmax &= target_mask

distance_markers, num_distance_markers = ndi.label(
    distance_hmax
)

if num_distance_markers == 0:
    raise RuntimeError(
        "No markers were detected. "
        "Try reducing H_MAXIMA_HEIGHT."
    )


# ------------------------------------------------------------
# 4. Full 3D distance-only watershed
# ------------------------------------------------------------

distance_watershed_labels = watershed(
    -distance_smooth,
    markers=distance_markers,
    mask=target_mask,
).astype(np.int32)


# ------------------------------------------------------------
# 5. Marker centers
# ------------------------------------------------------------

marker_ids = np.arange(
    1,
    num_distance_markers + 1,
)

marker_centers_zyx = ndi.center_of_mass(
    distance_smooth,
    labels=distance_markers,
    index=marker_ids,
)

marker_centers_zyx = np.asarray(
    marker_centers_zyx,
    dtype=float,
).reshape(-1, 3)


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

result_ids = np.unique(
    distance_watershed_labels
)

result_ids = result_ids[result_ids != 0]

print("Maximum physical distance:", float(distance_3d.max()))
print("Detected markers:", num_distance_markers)
print("Watershed regions:", len(result_ids))

print("\nMarker centers (Z, Y, X):")
print(np.round(marker_centers_zyx, 2))

print("\nRegion volumes:")

voxel_physical_volume = float(
    np.prod(VOXEL_SIZE_ZYX)
)

for region_id in result_ids:
    voxel_count = int(
        np.count_nonzero(
            distance_watershed_labels == region_id
        )
    )

    print(
        f"Region {region_id}: "
        f"{voxel_count} voxels, "
        f"{voxel_count * voxel_physical_volume:.2f} µm³"
    )

if num_distance_markers == 2:
    marker_distance = np.linalg.norm(
        (
                marker_centers_zyx[0]
                - marker_centers_zyx[1]
        )
        * voxel_size
    )

    print(
        "\nPhysical marker separation:",
        f"{marker_distance:.3f} µm",
    )

Distance smoothing sigma in voxels: (np.float64(0.492), np.float64(1.969), np.float64(1.969))
Maximum physical distance: 3.494694639736067
Detected markers: 2
Watershed regions: 2

Marker centers (Z, Y, X):
[[ 4. 26. 21.]
 [ 8. 24. 31.]]

Region volumes:
Region 1: 1242 voxels, 333.09 µm³
Region 2: 722 voxels, 193.63 µm³

Physical marker separation: 7.708 µm


In [42]:
heatmap_layer_name = "Distance heatmap 3D"

if heatmap_layer_name in viewer.layers:
    viewer.layers.remove(viewer.layers[heatmap_layer_name])

distance_heatmap_layer = viewer.add_image(
    distance_smooth.astype(np.float32),
    name=heatmap_layer_name,
    scale=VOXEL_SIZE_ZYX,
    colormap="turbo",   # you can also try "magma", "viridis", "inferno"
    rendering="mip",    # try "attenuated_mip" too
    opacity=0.9,
    visible=True,
)

viewer.layers["Original intensity"].visible = False
viewer.layers["Current watershed labels"].visible = False

viewer.reset_view()